# Complete SDS1 Analysis: Exploring Factor Effects

This tutorial demonstrates how to analyze a fully replicated factorial design (SDS 1) using the `by` parameter to explore different views of the same data.

## What You'll Learn

1. Load and formulate a two-factor study
2. Use the `by` parameter to aggregate Xbar/S charts at different levels
3. Stratify IMR charts by factor combinations
4. Understand lane boundaries in collapsed charts
5. Chart residuals using the `value` parameter

## Setup

In [1]:
from processbehavior import ProcessBehavior

## 1. Load and Formulate

We'll use the SDS1 validation dataset which has:
- **factor 1**: 3 levels (F1_1, F1_2, F1_3)
- **factor 2**: 2 levels (F2_1, F2_2)
- **time**: 8 periods
- Multiple replicates per cell

In [2]:
# Load the SDS1 validation data
pb = ProcessBehavior.read_csv('../../validation/sds1_data.csv')

print(f"Dataset: {pb.data.shape[0]} observations")
print(f"Factor 1 levels: {pb.data['factor 1'].unique().tolist()}")
print(f"Factor 2 levels: {pb.data['factor 2'].unique().tolist()}")
print(f"Time periods: {sorted(pb.data['time'].unique())}")
pb.data.head()

Dataset: 161 observations
Factor 1 levels: ['F1_1', 'F1_2', 'F1_3']
Factor 2 levels: ['F2_1', 'F2_2']
Time periods: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]


,time,factor 1,factor 2,y
0,1,F1_1,F2_1,51.346824
1,1,F1_1,F2_1,51.516211
2,2,F1_1,F2_1,52.666367
3,2,F1_1,F2_1,52.699215
4,2,F1_1,F2_1,52.242386


In [3]:
# Formulate the study with both factors
study = pb.formulate(
    response='y',
    factors=['factor 1', 'factor 2'],
    time='time'
)

print(f"SDS: {study.sds} ({study.sds_name})")
print(f"Valid charts: {study.valid_charts}")
print(f"Available residuals: {study.available_residuals}")

SDS: 1 (Full Factorial with Complete Replication)
Valid charts: ['Xbar', 'S', 'R', 'Imr', 'Histogram']
Available residuals: ['R2', 'R3', 'R4', 'R5']


## 1.5 Quick Distribution Check: Histogram

Before diving into control charts, you can visualize the distribution of your response variable using a histogram.

In [4]:
# Histogram of response variable
study.execute(chart='Histogram', bins=15).plot(template='ggplot').show()

In [13]:
study.execute(chart='Histogram', by=['factor 1'], bins=25).plot(template='ggplot').show()


In [10]:
study.execute(chart='Histogram', by=['factor 2'], bins=15).plot(template='ggplot').show()

You can also:
- Customize bins: `study.execute(chart='Histogram', bins=20)`
- Stratify by factors: `study.execute(chart='Histogram', by=['factor 1'])`
- Plot residual distributions: `study.execute(chart='Histogram', value='R5')`

## 2. Xbar Charts - Factor Aggregation

The `by` parameter controls how data points are aggregated on Xbar charts:
- **Default (all factors)**: One point per factor combination (6 points)
- **Single factor**: Aggregate across the other factor
- **Empty list**: Collapse to grand mean (1 point)

### 2.1 Xbar by All Factors (Default)

In [5]:
# Default: aggregate by all factors
result = study.execute(chart='Xbar')

print("Xbar chart data (one point per factor combination):")
result.get_chart('Xbar')

Xbar chart data (one point per factor combination):


,group,xbar,center,lpl,upl,beyond_limits
0,F1_1_F2_1_1,51.432,49.116,48.241,49.991,1
1,F1_1_F2_1_2,52.506,49.116,48.646,49.586,1
2,F1_1_F2_1_3,50.831,49.116,48.473,49.759,1
3,F1_1_F2_1_4,52.365,49.116,48.473,49.759,1
4,F1_1_F2_1_5,50.845,49.116,48.241,49.991,1
5,F1_1_F2_1_6,53.251,49.116,48.580,49.652,1
6,F1_1_F2_1_7,52.631,49.116,48.241,49.991,1
7,F1_1_F2_1_8,51.902,49.116,48.241,49.991,1
8,F1_1_F2_2_1,46.081,49.116,48.646,49.586,-1
9,F1_1_F2_2_2,47.795,49.116,48.241,49.991,-1


In [14]:
result.plot(chart='Xbar', show_stats=True).show()

### 2.2 Xbar by Factor 1 Only

In [15]:
# Aggregate by factor 1 only (3 points, one per F1 level)
result_f1 = study.execute(chart='Xbar', by=['factor 1'])

print("Xbar aggregated by factor 1:")
result_f1.get_chart('Xbar')

Xbar aggregated by factor 1:


,factor 1,xbar,center,lpl,upl,beyond_limits
0,F1_1,49.592,49.116,48.090,50.142,0
1,F1_2,47.380,49.116,48.136,50.096,-1
2,F1_3,50.543,49.116,48.090,50.142,1


In [17]:
result_f1.plot(chart='Xbar', show_stats=True).show()

### 2.3 Xbar by Factor 2 Only

In [19]:
# Aggregate by factor 2 only (2 points, one per F2 level)
result_f2 = study.execute(chart='Xbar', by=['factor 2'])

print("Xbar aggregated by factor 2:")
result_f2.get_chart('Xbar')

Xbar aggregated by factor 2:


,factor 2,xbar,center,lpl,upl,beyond_limits
0,F2_1,51.257,49.116,48.528,49.704,1
1,F2_2,46.949,49.116,48.524,49.708,-1


In [20]:
result_f2.plot(chart='Xbar', show_stats=True).show()

### 2.4 Xbar Collapsed (Grand Mean)

In [21]:
# Collapse all factors (single point - grand mean)
result_all = study.execute(chart='Xbar', by=[])

print("Xbar collapsed to grand mean:")
result_all.get_chart('Xbar')

Xbar collapsed to grand mean:


,group,xbar,center,lpl,upl,beyond_limits
0,All,49.116,49.116,48.457,49.775,0


## 3. S Charts - Variation Analysis

S charts follow the same `by` parameter logic as Xbar charts.

In [22]:
# S chart by all factors (default)
result_s = study.execute(chart='S')

print("S chart (within-group standard deviation):")
result_s.get_chart('S')

S chart (within-group standard deviation):


,group,s,center,lpl,upl,beyond_limits
0,F1_1_F2_1_1,0.120,0.329,-0.417,1.075,0
1,F1_1_F2_1_2,0.263,0.329,-0.029,0.688,0
2,F1_1_F2_1_3,0.072,0.329,-0.187,0.845,0
3,F1_1_F2_1_4,0.111,0.329,-0.187,0.845,0
4,F1_1_F2_1_5,0.132,0.329,-0.417,1.075,0
5,F1_1_F2_1_6,0.191,0.329,-0.088,0.746,0
6,F1_1_F2_1_7,0.476,0.329,-0.417,1.075,0
7,F1_1_F2_1_8,0.136,0.329,-0.417,1.075,0
8,F1_1_F2_2_1,0.177,0.329,-0.029,0.688,0
9,F1_1_F2_2_2,0.068,0.329,-0.417,1.075,0


In [23]:
result_s.plot(chart='S', show_stats=True).show()

In [24]:
# S chart by factor 1 only
result_s_f1 = study.execute(chart='Xbar', by=['factor 1'])

print("S chart aggregated by factor 1:")
result_s_f1.get_chart('S')

S chart aggregated by factor 1:


,factor 1,s,center,lpl,upl,beyond_limits
0,F1_1,2.457,2.455,1.724,3.186,0
1,F1_2,2.383,2.455,1.757,3.152,0
2,F1_3,2.524,2.455,1.724,3.186,0


In [25]:
# S chart by factor 2 only
result_s_f2 = study.execute(chart='Xbar', by=['factor 2'])

print("S chart aggregated by factor 2:")
result_s_f2.get_chart('S')

S chart aggregated by factor 2:


,factor 2,s,center,lpl,upl,beyond_limits
0,F2_1,1.848,1.759,1.341,2.177,0
1,F2_2,1.669,1.759,1.338,2.179,0


## 4. IMR Charts - Stratified Analysis

IMR charts with factors **require** an explicit `by` parameter. The `by` parameter controls stratification:
- **Both factors**: Separate chart for each factor combination
- **Single factor**: Charts per level with lane boundaries showing the other factor
- **Empty list**: Single chart with lane boundaries for all factor transitions

### 4.1 IMR by Both Factors (6 Faceted Charts)

In [26]:
# IMR stratified by both factors
result_imr = study.execute(chart='Imr', by=['factor 1', 'factor 2'])

print(f"Strata: {result_imr.charts['Imr']['strata']}")
print(f"Each stratum has its own IMR chart")

Strata: [('F1_1', 'F2_1'), ('F1_1', 'F2_2'), ('F1_2', 'F2_1'), ('F1_2', 'F2_2'), ('F1_3', 'F2_1'), ('F1_3', 'F2_2')]
Each stratum has its own IMR chart


In [27]:
result_imr.plot(chart='Imr', show_zones=True).show()

### 4.2 IMR by Factor 1 Only (3 Charts with Lane Boundaries)

When stratifying by one factor, the collapsed factor creates multiple observations at each time point. **Lane boundaries** show where the collapsed factor changes.

In [28]:
# IMR stratified by factor 1 only
result_imr_f1 = study.execute(chart='Imr', by=['factor 1'])

print(f"Strata: {result_imr_f1.charts['Imr']['strata']}")
print("\nLane boundaries show where factor 2 changes within each chart")

Strata: ['F1_1', 'F1_2', 'F1_3']

Lane boundaries show where factor 2 changes within each chart


In [30]:
result_imr_f1.plot(chart='Imr', show_zones=True).show()

### 4.3 IMR by Factor 2 Only (2 Charts with Lane Boundaries)

In [32]:
# IMR stratified by factor 2 only
result_imr_f2 = study.execute(chart='Imr', by=['factor 2'])

print(f"Strata: {result_imr_f2.charts['Imr']['strata']}")
print("\nLane boundaries show where factor 1 changes within each chart")

Strata: ['F2_1', 'F2_2']

Lane boundaries show where factor 1 changes within each chart


In [33]:
result_imr_f2.plot(chart='Imr', show_zones=True).show()

### 4.4 Single IMR (Collapsed, with Lane Boundaries)

In [35]:
# Single IMR chart with all factors collapsed
result_imr_all = study.execute(chart='Imr', by=[])

print("Single IMR chart with all data")
print("Lane boundaries show transitions between factor combinations")

Single IMR chart with all data
Lane boundaries show transitions between factor combinations


In [36]:
result_imr_all.plot(chart='Imr', show_zones=True).show()

## 5. Residual Charts

Use the `value` parameter to chart VAS residuals instead of the response variable.

### 5.1 R5 (Factor Effects) on Xbar

In [37]:
# R5 residuals show factor effects
result_r5 = study.execute(chart='Xbar', value='R5')

print("R5 Xbar chart (factor effects):")
result_r5.get_chart('Xbar')

R5 Xbar chart (factor effects):


,group,xbar,center,lpl,upl,beyond_limits
0,F1_1_F2_1_1,3.003,0.0,-0.875,0.875,1
1,F1_1_F2_1_2,3.003,0.0,-0.470,0.470,1
2,F1_1_F2_1_3,3.003,0.0,-0.643,0.643,1
3,F1_1_F2_1_4,3.003,0.0,-0.643,0.643,1
4,F1_1_F2_1_5,3.003,0.0,-0.875,0.875,1
5,F1_1_F2_1_6,3.003,0.0,-0.536,0.536,1
6,F1_1_F2_1_7,3.003,0.0,-0.875,0.875,1
7,F1_1_F2_1_8,3.003,0.0,-0.875,0.875,1
8,F1_1_F2_2_1,-1.528,0.0,-0.470,0.470,-1
9,F1_1_F2_2_2,-1.528,0.0,-0.875,0.875,-1


In [38]:
result_r5.plot(chart='Xbar', show_stats=True).show()

### 5.2 Recentered Residuals

Use `recentered=True` to center residuals around zero.

In [39]:
# Recentered R5 residuals
result_r5_rc = study.execute(chart='Xbar', value='R5', recentered=True)

print("Recentered R5 Xbar chart:")
result_r5_rc.get_chart('Xbar')

Recentered R5 Xbar chart:


,group,xbar,center,lpl,upl,beyond_limits
0,F1_1_F2_1_1,51.432,49.116,48.241,49.991,1
1,F1_1_F2_1_2,52.506,49.116,48.646,49.586,1
2,F1_1_F2_1_3,50.831,49.116,48.473,49.759,1
3,F1_1_F2_1_4,52.365,49.116,48.473,49.759,1
4,F1_1_F2_1_5,50.845,49.116,48.241,49.991,1
5,F1_1_F2_1_6,53.251,49.116,48.580,49.652,1
6,F1_1_F2_1_7,52.631,49.116,48.241,49.991,1
7,F1_1_F2_1_8,51.902,49.116,48.241,49.991,1
8,F1_1_F2_2_1,46.081,49.116,48.646,49.586,-1
9,F1_1_F2_2_2,47.795,49.116,48.241,49.991,-1


In [40]:
result_r5_rc.plot(chart='Xbar', show_stats=True).show()

### 5.3 R5 on S Chart

In [44]:
# R5 on S chart

study.execute(chart='S', value='R5', recentered=True).plot().show()

## Summary

### When to Use Each `by` Configuration

| Configuration | Use Case |
|--------------|----------|
| `by=None` (default) | Compare all factor combinations |
| `by=['factor 1']` | Focus on one factor, aggregate the other |
| `by=[]` | Overall process view, collapsed factors |
| `by=['factor 1', 'factor 2']` | Individual charts per combination |

### Key Concepts

1. **Views, Not Recomputation**: The `by` parameter creates views over the same underlying data. Residuals never change.

2. **Lane Boundaries**: When IMR charts collapse factors, vertical lane boundaries show where factor transitions occur.

3. **Residuals via `value`**: Use `value='R5'` to chart residuals instead of response.

4. **Recentering**: Use `recentered=True` to center residuals around zero.